# RAG from Scratch to Production — Episode 7/20
## BM25: keyword search, done properly

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/JakkulaMahendhar/rag-from-scratch/blob/main/notebooks/07_bm25.ipynb)

Episode 5 showed embeddings matching meaning: "training money" found the *learning budget*. But company documents are full of things that have no "meaning" to match: **form numbers, error codes, product names**. For those, you want the exact word.

That's **BM25**, the keyword ranking behind most search engines, and half of the [production app](https://github.com/JakkulaMahendhar/rag-chatbot)'s hybrid search.

**In this episode:**
1. Build BM25 from scratch in ~15 lines, and prove it matches the `rank-bm25` library production uses.
2. See the 3 ideas that make it better than Episode 1's word counting.
3. Find a **one-character bug** in how production splits text into words.
4. Race embeddings vs BM25 on codes and on everyday wording.

**No API key needed.**

## 0. Setup

In [ ]:
%pip install -q rank-bm25 sentence-transformers

In [1]:
import re, math
import numpy as np
from rank_bm25 import BM25Okapi
from sentence_transformers import SentenceTransformer

## 1. A handbook with exact terms

The NovaTech handbook, now with what real company documents are full of: **form numbers** (HR-12, HR-17, EXP-22), **error codes** (E-4031, E-4032) and **product names** (ThinkPad, Okta).

In [2]:
CHUNKS = [
    "Remote Work\nEmployees may work remotely up to 2 days per week. Wednesday is a mandatory in-office day for all teams. Remote days must be logged in the HR portal by Friday of the previous week.",
    "Annual Leave\nFull-time employees receive 24 days of paid annual leave per calendar year. A maximum of 5 unused days may be carried over into the next year. Book leave with form HR-12.",
    "Parental Leave\nNew parents receive 16 weeks of fully paid leave. Submit form HR-17 at least 8 weeks before the expected start date.",
    "Expenses\nThe meal allowance during business travel is capped at 40 USD per day. Travel expenses are claimed with form EXP-22 within 30 days. Claims above 500 USD need the finance director.",
    "Learning Budget\nEvery employee has an annual learning budget of 1,200 USD for courses and conferences. It does not roll over. Requests are approved by your direct manager.",
    "Equipment\nEvery employee receives a ThinkPad X1 Carbon laptop, refreshed every 3 years. Lost or stolen devices must be reported to IT within 24 hours.",
    "IT Support\nIf the VPN shows error E-4031, your access token has expired: reset it in the Okta portal. For error E-4032, contact the IT helpdesk.",
    "Security\nMulti-factor authentication is mandatory on all company accounts. Passwords must be at least 14 characters long and must never be reused.",
]

def title(chunk: str) -> str:
    return chunk.split("\n")[0]

print([title(c) for c in CHUNKS])

['Remote Work', 'Annual Leave', 'Parental Leave', 'Expenses', 'Learning Budget', 'Equipment', 'IT Support', 'Security']


## 2. BM25 from scratch

Episode 1's keyword search just counted shared words. BM25 adds three ideas:

| Idea | In plain words | Knob |
|---|---|---|
| **Rare words count more** | Matching "parental" (in 1 chunk) is strong evidence. Matching "the" (in every chunk) is none | IDF |
| **Repeats help less and less** | A chunk saying "leave" 5 times isn't 5× more relevant | `k1` (1.5) |
| **Long chunks don't win by size** | A long chunk matches more words just by being long, so scores are adjusted for length | `b` (0.75) |

First, split text into words (**tokens**):

In [3]:
def tokenize(text: str) -> list[str]:
    """Lowercase words and numbers. Keeps codes like 'hr-17' and 'e-4031' together."""
    return re.findall(r"[a-z0-9]+(?:-[a-z0-9]+)*", text.lower())

print(tokenize("Book leave with form HR-12."))

['book', 'leave', 'with', 'form', 'hr-12']


Now BM25 itself:

In [4]:
class BM25:
    def __init__(self, chunks: list[str], k1: float = 1.5, b: float = 0.75, epsilon: float = 0.25):
        self.docs = [tokenize(c) for c in chunks]
        self.k1, self.b = k1, b
        self.avg_len = sum(map(len, self.docs)) / len(self.docs)

        # IDF: how rare is each word? (in how many chunks does it appear)
        n = len(self.docs)
        appears_in = {}
        for doc in self.docs:
            for word in set(doc):
                appears_in[word] = appears_in.get(word, 0) + 1
        self.idf = {w: math.log((n - k + 0.5) / (k + 0.5)) for w, k in appears_in.items()}
        # Words in more than half the chunks get a negative IDF: floor them at a small positive value
        floor = epsilon * sum(self.idf.values()) / len(self.idf)
        self.idf = {w: (v if v >= 0 else floor) for w, v in self.idf.items()}

    def score(self, query: str) -> list[float]:
        scores = []
        for doc in self.docs:
            length_factor = 1 - self.b + self.b * len(doc) / self.avg_len
            s = 0.0
            for word in tokenize(query):
                tf = doc.count(word)                       # how often the word is in this chunk
                s += self.idf.get(word, 0) * tf * (self.k1 + 1) / (tf + self.k1 * length_factor)
            scores.append(s)
        return scores


bm25 = BM25(CHUNKS)

**Is it right?** Compare it with `rank-bm25` (the library the production app uses) on a few questions:

In [5]:
library = BM25Okapi([tokenize(c) for c in CHUNKS])

for q in ["How many remote days per week?", "What is form EXP-22 for?", "VPN error E-4031"]:
    ours, theirs = bm25.score(q), library.get_scores(tokenize(q))
    print(f"{q:<34} identical: {np.allclose(ours, theirs)}")

How many remote days per week?     identical: True
What is form EXP-22 for?           identical: True
VPN error E-4031                   identical: True


Identical. Here are the IDF values behind it: rare words carry the score.

In [6]:
for word in ["parental", "hr-17", "e-4031", "leave", "form", "employee", "the"]:
    in_chunks = sum(word in doc for doc in bm25.docs)
    print(f"{word:<10} in {in_chunks} of {len(CHUNKS)} chunks   IDF {bm25.idf.get(word, 0):5.2f}")

parental   in 1 of 8 chunks   IDF  1.61
hr-17      in 1 of 8 chunks   IDF  1.61
e-4031     in 1 of 8 chunks   IDF  1.61
leave      in 2 of 8 chunks   IDF  0.96
form       in 3 of 8 chunks   IDF  0.45
employee   in 2 of 8 chunks   IDF  0.96
the        in 5 of 8 chunks   IDF  0.35


A code like "hr-17" appears in just one chunk, so it's worth the most (1.61). "the" appears in 5 of 8 chunks: its raw IDF is negative, so BM25 floors it at a small value (0.35).

## 3. The one-character bug

The production app splits text into words like this:

```python
tokens = text.lower().split()
```

`.split()` splits on spaces only. Punctuation stays glued to the word:

In [7]:
def tokenize_production(text: str) -> list[str]:
    return text.lower().split()

print("question:", tokenize_production("Which form is HR-12?"))
print("chunk:   ", [t for t in tokenize_production(CHUNKS[1]) if "hr" in t])

question: ['which', 'form', 'is', 'hr-12?']
chunk:    ['hr-12.']


The question has **`hr-12?`**, the chunk has **`hr-12.`**. To BM25 those are different words, so the most important word in the question matches nothing.

Users end questions with `?`. Documents end sentences with `.`. So the last word of almost every question is broken.

In [8]:
production_bm25 = BM25Okapi([tokenize_production(c) for c in CHUNKS])

def best(scores) -> str:
    i = int(np.argmax(scores))
    return title(CHUNKS[i]) if scores[i] > 0 else "(nothing)"

for q in ["Which form is HR-12?", "Which form do I need for parental leave? Is it HR-17?"]:
    print(f"Q: {q}")
    print(f"   production tokenizer → {best(production_bm25.get_scores(tokenize_production(q)))}")
    print(f"   fixed tokenizer      → {best(bm25.score(q))}\n")

Q: Which form is HR-12?
   production tokenizer → Expenses
   fixed tokenizer      → Annual Leave

Q: Which form do I need for parental leave? Is it HR-17?
   production tokenizer → Expenses
   fixed tokenizer      → Parental Leave



The parental leave question is the worst case. Here's why it lands on *Expenses*:

In [9]:
q = "Which form do I need for parental leave? Is it HR-17?"
idf = production_bm25.idf
print("word weights with the production tokenizer:")
print({w: round(idf.get(w, 0), 2) for w in tokenize_production(q)})
print()
for chunk, s in zip(CHUNKS, production_bm25.get_scores(tokenize_production(q))):
    print(f"   {s:5.2f}  {title(chunk)}")

word weights with the production tokenizer:
{'which': 0, 'form': 0.45, 'do': 0, 'i': 0, 'need': 1.61, 'for': 0.45, 'parental': 1.61, 'leave?': 0, 'is': 0.45, 'it': 0.45, 'hr-17?': 0}

    0.80  Remote Work
    0.42  Annual Leave
    2.25  Parental Leave
    2.37  Expenses
    0.91  Learning Budget
    0.48  Equipment
    1.24  IT Support
    0.50  Security


`leave?` and `hr-17?` are worth **0**: they don't exist in any chunk. That leaves generic words ("need", "form", "for") to decide, and *Expenses* edges out *Parental Leave*.

**The fix is one line:** tokenize with a regular expression instead of `.split()`, and use the **same** tokenizer for chunks and questions.

## 4. The race: embeddings vs BM25

11 questions in two groups: **exact terms** (codes, forms, product names) and **everyday wording** (from Episode 5). Three retrievers:
- **Embeddings** (Episode 5's model, the same as production)
- **BM25 with production's tokenizer**
- **BM25 with the fixed tokenizer**

In [10]:
model = SentenceTransformer("all-MiniLM-L6-v2")
chunk_vectors = model.encode(CHUNKS, normalize_embeddings=True)

def search_embeddings(q):  return best(chunk_vectors @ model.encode(q, normalize_embeddings=True))
def search_bm25_prod(q):   return best(production_bm25.get_scores(tokenize_production(q)))
def search_bm25_fixed(q):  return best(bm25.score(q))

QUESTIONS = [
    # exact terms
    ("What is form EXP-22 for?",                               "Expenses"),
    ("Which form do I need for parental leave? Is it HR-17?",  "Parental Leave"),
    ("VPN says E-4031, what do I do?",                         "IT Support"),
    ("What does error E-4032 mean?",                           "IT Support"),
    ("Which form is HR-12?",                                   "Annual Leave"),
    ("What laptop model do we get, is it a ThinkPad?",         "Equipment"),
    ("How do I reset my Okta token?",                          "IT Support"),
    # everyday wording
    ("Can I save unused training money for next year?",        "Learning Budget"),
    ("Do I need 2FA on my work accounts?",                     "Security"),
    ("What do I get for food when travelling for work?",       "Expenses"),
    ("How many remote days per week?",                         "Remote Work"),
]

RETRIEVERS = {"embeddings": search_embeddings, "BM25 (prod)": search_bm25_prod, "BM25 (fixed)": search_bm25_fixed}
totals = {name: 0 for name in RETRIEVERS}

print(f"{'question':<46}" + "".join(f"{name:<15}" for name in RETRIEVERS))
for i, (q, expected) in enumerate(QUESTIONS):
    if i == 7:
        print("── everyday wording")
    row = ""
    for name, search in RETRIEVERS.items():
        hit = search(q) == expected
        totals[name] += hit
        row += f"{'✅' if hit else '❌'}{'':<13}"
    print(f"{q[:44]:<46}{row}")
print(f"\n{'TOTAL (of 11)':<46}" + "".join(f"{totals[n]:<15}" for n in RETRIEVERS))

question                                      embeddings     BM25 (prod)    BM25 (fixed)   
What is form EXP-22 for?                      ❌             ✅             ✅             
Which form do I need for parental leave? Is   ✅             ❌             ✅             
VPN says E-4031, what do I do?                ✅             ✅             ✅             
What does error E-4032 mean?                  ✅             ✅             ✅             
Which form is HR-12?                          ✅             ❌             ✅             
What laptop model do we get, is it a ThinkPa  ✅             ✅             ✅             
How do I reset my Okta token?                 ✅             ✅             ✅             
── everyday wording
Can I save unused training money for next ye  ✅             ❌             ❌             
Do I need 2FA on my work accounts?            ✅             ❌             ✅             
What do I get for food when travelling for w  ✅             ❌             ❌            

What the table shows:
- **The tokenizer fix** takes BM25 from 6 to 9 out of 11, with no other change.
- **BM25 (fixed) gets every exact term**, including EXP-22, which embeddings miss.
- **Embeddings win on everyday wording**: "training money" and "food when travelling" share no useful words with the right sections.
- Embeddings did better on codes than you might expect (6 of 7 here). But when they miss an exact term, nothing else would catch it.

**They fail on different questions.** Check it:

In [11]:
both = [q for q, expected in QUESTIONS
        if search_embeddings(q) == expected or search_bm25_fixed(q) == expected]
print(f"Found by at least one of them: {len(both)} of {len(QUESTIONS)}")

Found by at least one of them: 11 of 11


Together they'd cover every question. That's the idea behind **hybrid search**, Episode 8.

## 5. Production notes

The [production app](https://github.com/JakkulaMahendhar/rag-chatbot)'s BM25 (`app/services/bm25_search.py`):
- Uses `rank-bm25`'s `BM25Okapi`, exactly the formula above.
- Tokenizes with `.lower().split()`, the bug from section 3.
- Builds **one index over every user's chunks**, then filters the results by the user's documents. So word rarity (IDF) is computed across all users: how much "hr-17" is worth for user 1 depends on user 2's files too.
- Rebuilds the whole index on every upload and saves it to disk with `pickle`. Fine for thousands of chunks. At much larger scale you'd use a search engine (Elasticsearch, OpenSearch) or a database with built-in full-text search.

## Recap

| | Embeddings | BM25 |
|---|---|---|
| Matches | Meaning | Exact words |
| "training money" → learning budget | ✅ | ❌ |
| "EXP-22" → Expenses | ❌ | ✅ |
| Needs | A model, run on every chunk | Just word counts |
| Biggest trap | Unknown abbreviations (Episode 5) | **Tokenization**: `hr-12?` ≠ `hr-12.` |

**The rule:** keyword search is only as good as its tokenizer. Use the **same** tokenizer for chunks and questions, and strip punctuation.

**Next, Episode 8:** embeddings and BM25 fail on different questions. **Hybrid search** combines them, and getting the combination right is harder than it looks.

⭐ Notebooks: https://github.com/JakkulaMahendhar/rag-from-scratch
🏗️ Full production app: https://github.com/JakkulaMahendhar/rag-chatbot
📸 Instagram: [@codingwithmahi](https://instagram.com/codingwithmahi)